# DS05 · Missing values, selection, and train-only imputation

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PD02](../../curriculum/papers/design.md#pd02), [PM03](../../curriculum/papers/modeling.md#pm03).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** Whose observations disappear when values are missing, and who is allowed to inform an imputation rule?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Format:** 75–100 minutes of guided work, plus 30–60 minutes in the assigned existing course material. **Prerequisite:** the foundations notebooks; follow this strand in order. The core exercise is synthetic, offline, and independently runnable. It demonstrates mechanics, not a validated participant-data analysis.

## Existing course material

Read [Data 8: Sampling from a population; supplement: sklearn imputation](https://github.com/data-8/textbook/blob/5235b7653f8dfaeb90e43419b9aa069322f2d60b/chapters/10/2/Sampling_from_a_Population.ipynb). Use the indicated topic, then return here to apply it to a neuroimaging question. Berkeley material is linked in its original form, not adapted or redistributed; its CC BY-NC-ND terms remain upstream. Neuromatch material is CC BY 4.0 with separately licensed software; selected unmodified copies live in `third_party/data_science`. The explanation and dataset below are original.

## Understand the transformation

Missingness is a mechanism, not merely a special value. A randomly missing observation may have different consequences from a missing scan caused by severe motion associated with the outcome. Complete-case analysis can alter the composition of the sample. Filling missing values with zero invents a measurement and can create false group differences when missing rates vary.

We distinguish missing completely at random, missing at random conditional on observed variables, and missing not at random. These are assumptions about how missingness relates to data, not labels AI can reliably infer from a small table. We simulate a mechanism that preferentially hides high outcomes to show how the observed mean can be biased. The correct response is to investigate and model the mechanism, not assume imputation automatically restores the population.

For predictive features, an imputer's parameters must be fitted on training data and reused on validation/test data. Target imputation is a separate and usually inappropriate shortcut for supervised evaluation: a guessed outcome is not observed truth. This lab uses simple mean imputation only to expose the fit boundary. A real study may need multiple imputation with a justified model, missingness indicators, sensitivity analyses, or restricted claims. The stored missingness mask and the raw table remain part of the provenance.

## AI-guided prediction

First answer in your own words; then send this to Goose/Ollama or ChatGPT:

> Ask me whether values are unobserved or truly zero. Simulate outcome-dependent missingness, show the observed sample shift, then demonstrate training-only feature imputation. Clearly separate the selection problem from the mechanical fill operation.

Use the model as a tutor and snippet writer. Require it to name the axes, units, fitting population, expected output, and one failure check. A code cell that runs is not proof that it answers the scientific question. Keep raw data unchanged and save your actual settings.

## Experiment

Hide the largest fifth of a synthetic outcome. Compare observed and complete means. Fit an imputer to [1,3,missing] and apply it to [100,missing]; deliberately include test data in the fit and compare fill values.

Run the following cells in order. Before each, predict what should remain unchanged and what should differ. The assertions test specific mathematical or bookkeeping properties, not clinical validity.

In [1]:
import numpy as np
from sklearn.impute import SimpleImputer
rng=np.random.default_rng(305); y=rng.normal(size=10000)
observed=y <= np.quantile(y,.8)
assert y[observed].mean() < y.mean()-.2
train=np.array([[1.],[3.],[np.nan]]); test=np.array([[100.],[np.nan]])
imputer=SimpleImputer().fit(train)
filled=imputer.transform(test)
leaky=SimpleImputer().fit(np.vstack([train,test]))
assert filled[1,0] == 2
assert leaky.statistics_[0] != imputer.statistics_[0]
print('Complete/observed mean:',y.mean(),y[observed].mean())
print('Train-only/leaky fill:',imputer.statistics_,leaky.statistics_)

Complete/observed mean: 0.003626182017483322 -0.3477255120516691
Train-only/leaky fill: [2.] [34.66666667]


## Explain, break, transfer

1. Save an input → operation → output diagram and state what information was lost.
2. Make the specified wrong choice above. Compare its result with the reference checks; explain why the misleading result is possible.
3. Work through the assigned upstream chapter's example using its own environment or hosted reader. Record one difference between its data and a participant/voxel/time-series dataset.
4. Ask the AI for a short application to a real imaging table, but do not run it until participant identifiers, units, missingness, and any training/test boundary are explicit. Never infer those properties from the column names alone.

**Evidence to submit:** one labeled result, the changed parameter, a failure diagnosis, and a five-sentence interpretation that separates a computational check from the research claim. Explain the result without looking at the model's wording.

<details><summary>Instructor check / answer guide</summary>

The observed mean is lower because high values were selectively hidden. Training-only fill is 2; fitting all data contaminates this value. Neither result resolves the outcome-dependent selection mechanism.

</details>

**Scope:** This local notebook and its numerical checks are part of the executable core. Completion of the external chapter is a learner assignment; its execution is not implied by the local result. No endorsement by the source authors or USC is implied.

### Return to the research question

Reopen [PD02](../../curriculum/papers/design.md#pd02), [PM03](../../curriculum/papers/modeling.md#pm03) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
